# 1.0 Import Libraries & Data

In [35]:
import pandas as pd
from extra.utils import load_config
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

In [36]:
cfg = load_config()

Config loaded successfully!

data:
  raw_path: ../data/raw/smart_manufacturing_data.csv
  interim_path: ../data/interim/cleaned.csv
  processed_path: ../data/processed/features.csv
  target_column: maintenance_required
cleaning: null
features:
  rolling_window_size: 5
  sensor_columns:
  - temperature
  - vibration
  - humidity
  - pressure
  - energy_consumption
model:
  n_jobs: 2
  session_id: 42
  train_size: 0.8
  ignore_features:
  - failure_type
  - downtime_risk
  - anomaly_flag
  - machine_status
  - predicted_remaining_life
mlflow:
  tracking_uri: http://127.0.0.1:5000
  experiment_name: Smart Manufacturing Maintenance



In [37]:
df = pd.read_csv(cfg.data.interim_path, parse_dates=["timestamp"])

In [38]:
# Double check data type is OK before feature engineering
df.dtypes

timestamp                   datetime64[ns]
machine_id                           int64
temperature                        float64
vibration                          float64
humidity                           float64
pressure                           float64
energy_consumption                 float64
machine_status                       int64
anomaly_flag                         int64
predicted_remaining_life             int64
failure_type                        object
downtime_risk                      float64
maintenance_required                 int64
dtype: object

# 2.0 Feature Engineering
In the section, I create new features in order to provide more information to the machine learning models. I enrich my data by increasing the granularity of it, I either break down existing features into smaller pieces or combine them to create new interaction features. In this part I create 4 distinct kinds of features; Temporal, Statistical and Interactional. I will talk more about each one below.

In [39]:
# Sort first (required for all time-based calculations)
df = df.sort_values(["machine_id", "timestamp"]).reset_index(drop=True)

In [40]:
def show_new_features(df, previous_columns):
    current_columns = set(df.columns)
    previous_columns = set(previous_columns)

    new_columns = sorted(current_columns - previous_columns)

    if new_columns:
        print(f"Added {len(new_columns)} feature(s):")
        for col in new_columns:
            print(f"  - {col}")
    else:
        print("No new features added.")

    return df.columns.copy()

In [41]:
tracked_columns = df.columns.copy()

## 2.1 Time based Features
In this section, the timestamp feature is decomposed into four components: the hour of the day, day of the week, weekend indicator, and shift categorisation. These features allow the model to capture temporal patterns that may influence machine behaviour, such as differences between working hours and overnight periods or weekday and weekend operations. By extracting these meaningful time-based attributes, the model can better identify recurring operational trends that may be associated with maintenance requirements.

In [ ]:
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

def assign_shift(hour):
    if 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 19:
        return "Afternoon"
    else:
        return "Night"

df["shift"] = df["hour"].apply(assign_shift).astype("category")

In [43]:
tracked_columns = show_new_features(df, tracked_columns)

Added 4 feature(s):
  - day_of_week
  - hour
  - is_weekend
  - shift


## 2.2 Rolling Statistics & Rate of Change
The original dataset only provides the model with information at the current point in time and does not capture how sensor readings have changed over previous observations. Rolling statistics and rate of change features are therefore introduced to provide historical context, enabling the model to learn trends and detect gradual changes in machine behaviour that may precede maintenance events.

In [57]:
sensor_cols = [
    "temperature",
    "vibration",
    "humidity",
    "pressure",
    "energy_consumption"
]

for col in sensor_cols:
    df[f"{col}_rolling_mean"] = (
        df.groupby("machine_id")[col]
          .transform(lambda x: x.rolling(cfg.features.rolling_window_size, min_periods=1).mean())
    )

In [45]:
tracked_columns = show_new_features(df, tracked_columns)

Added 5 feature(s):
  - energy_consumption_rolling_mean
  - humidity_rolling_mean
  - pressure_rolling_mean
  - temperature_rolling_mean
  - vibration_rolling_mean


In [46]:
for col in sensor_cols:
    df[f"{col}_diff"] = (
        df.groupby("machine_id")[col]
          .diff()
          .fillna(0)
    )

In [47]:
tracked_columns = show_new_features(df, tracked_columns)

Added 5 feature(s):
  - energy_consumption_diff
  - humidity_diff
  - pressure_diff
  - temperature_diff
  - vibration_diff


## 2.3 Z-score Normalization
Z-score normalization is applied to standardize sensor readings based on each machine's historical behaviour. This allows the model to identify abnormal deviations from a machine's normal operating range, as different machines may naturally have different baseline sensor values.

In [48]:
for col in sensor_cols:
    machine_mean = df.groupby("machine_id")[col].transform("mean")
    machine_std = df.groupby("machine_id")[col].transform("std")

    df[f"{col}_zscore"] = (
        (df[col] - machine_mean) /
        machine_std.replace(0, 1)
    )

In [49]:
tracked_columns = show_new_features(df, tracked_columns)

Added 5 feature(s):
  - energy_consumption_zscore
  - humidity_zscore
  - pressure_zscore
  - temperature_zscore
  - vibration_zscore


## 2.4 Interaction Features
Interaction features are created by combining multiple sensor measurements to capture relationships that may not be obvious from individual features. These combined features allow the model to detect more complex patterns, such as when multiple sensor conditions occur together and contribute to increased maintenance risk. This part is more useful for models like Linear/Logistics Regression where they cannot learn interactions automatically unlike many tree-based models. (Non-essential columns)

In [50]:
df["temp_x_vibration"] = (
    df["temperature"] *
    df["vibration"]
)

df["temp_x_pressure"] = (
    df["temperature"] *
    df["pressure"]
)

df["vibration_x_pressure"] = (
    df["vibration"] *
    df["pressure"]
)

df["temp_x_energy"] = (
    df["temperature"] *
    df["energy_consumption"]
)

In [51]:
tracked_columns = show_new_features(df, tracked_columns)

Added 4 feature(s):
  - temp_x_energy
  - temp_x_pressure
  - temp_x_vibration
  - vibration_x_pressure


In [52]:
print("Final features in dataset")
df.columns

Final features in dataset


Index(['timestamp', 'machine_id', 'temperature', 'vibration', 'humidity',
       'pressure', 'energy_consumption', 'machine_status', 'anomaly_flag',
       'predicted_remaining_life', 'failure_type', 'downtime_risk',
       'maintenance_required', 'hour', 'day_of_week', 'is_weekend', 'shift',
       'temperature_rolling_mean', 'vibration_rolling_mean',
       'humidity_rolling_mean', 'pressure_rolling_mean',
       'energy_consumption_rolling_mean', 'temperature_diff', 'vibration_diff',
       'humidity_diff', 'pressure_diff', 'energy_consumption_diff',
       'temperature_zscore', 'vibration_zscore', 'humidity_zscore',
       'pressure_zscore', 'energy_consumption_zscore', 'temp_x_vibration',
       'temp_x_pressure', 'vibration_x_pressure', 'temp_x_energy'],
      dtype='object')

## 2.5 Feature Selection


In [53]:
filtered_df = df.copy()

leaky_and_nonfeature_cols = ['maintenance_required', 'machine_status', 'anomaly_flag', 'downtime_risk', 'predicted_remaining_life']

X = filtered_df.select_dtypes(include=np.number).drop(columns=leaky_and_nonfeature_cols, errors='ignore')
y = filtered_df["maintenance_required"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_results = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

print(mi_results)

temperature                        0.121152
temperature_zscore                 0.117269
vibration                          0.038109
temp_x_vibration                   0.033683
vibration_zscore                   0.032586
temperature_diff                   0.024842
temp_x_pressure                    0.012198
temp_x_energy                      0.011101
temperature_rolling_mean           0.009915
vibration_x_pressure               0.006250
is_weekend                         0.006247
vibration_diff                     0.005699
energy_consumption_rolling_mean    0.003122
day_of_week                        0.003036
machine_id                         0.001998
pressure_zscore                    0.001841
pressure_diff                      0.001499
pressure_rolling_mean              0.001373
humidity_zscore                    0.001269
vibration_rolling_mean             0.000879
energy_consumption_zscore          0.000685
hour                               0.000569
energy_consumption              

In [54]:
filtered_df = df.copy()

if os.path.exists("useful_features.txt"):
    with open("extra/useful_features.txt", "r") as f:
        useful_features = f.read().splitlines()
else:
    nonzero_mi = mi_results[mi_results > 0].sort_values(ascending=False)
    useful_features = ['timestamp'] + nonzero_mi.head(19).index.tolist()
    with open("extra/useful_features.txt", "w") as f:
        f.write("\n".join(useful_features))

filtered_df = filtered_df[useful_features + ["maintenance_required"]]

In [55]:
filtered_df.shape

(99360, 21)

In [59]:
print("Final features in filtered dataset")
filtered_df.columns

Final features in filtered dataset


Index(['timestamp', 'temperature', 'temperature_zscore', 'vibration',
       'temp_x_vibration', 'vibration_zscore', 'temperature_diff',
       'temp_x_pressure', 'temp_x_energy', 'temperature_rolling_mean',
       'vibration_x_pressure', 'is_weekend', 'vibration_diff',
       'energy_consumption_rolling_mean', 'day_of_week', 'machine_id',
       'pressure_zscore', 'pressure_diff', 'pressure_rolling_mean',
       'humidity_zscore', 'maintenance_required'],
      dtype='object')

In [60]:
filtered_df.to_csv(cfg.data.processed_path, index=False) # Output